## Collecting data about companies by web-scraping from ambition-box

### 7 features -- 

- Company name
- Company rating
- No of reviews
- Company Type
- Headquarter Location
- Critically Rated for
- Highly Rated for

In [17]:
import pandas as pd
import requests
from bs4 import BeautifulSoup


In [18]:
#### NOTE:

#Error -- 403, means that the server has rejected the request
#Ambition box has a robots.txt file created internally to prevent web-scraping
#So, request needs to be disguised in a manner that ambition box thinks that request is coming from browser by a human and not from a bot

#To disguise the request to look like its a human from a browser, use headers. create a headers dict with value = user agent(can be found by
#searching My user agent on google), key = 'User-Agent'. Then pass the headers dict as value of parameter - headers, in requests.get

#This goes to the url and gets its whole html code from page source!

In [19]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36'}
webpage = requests.get("https://www.ambitionbox.com/list-of-companies?page=1", headers=headers).text

In [20]:
#Now create a soup object, and give it the webpage along with the built in HTML parser (lxml)
soup = BeautifulSoup(webpage, 'lxml')
# print(soup.prettify()) #It makes the html more organized and easy to understand

In [21]:
#To print all the h1 tags in webpage--

soup.find_all('h1') #got list as an o/p, there is only one h1 tag in webpage

[<h1 class="companyListing__collectionTopHeadingCont" data-v-bb58b91c=""><div class="companyListing__collectionTopHeading" data-v-bb58b91c="">
 					Top Companies in
 				</div> <div class="companyListing__collectionIndiaHeading" data-v-bb58b91c=""><img data-v-bb58b91c="" height="12" src="https://static.ambitionbox.com/static/loc/loc-star.png" width="12"/>
 					INDIA
 					<img data-v-bb58b91c="" height="12" src="https://static.ambitionbox.com/static/loc/loc-star.png" width="12"/></div></h1>]

In [22]:
soup.find_all('h1')[0]

<h1 class="companyListing__collectionTopHeadingCont" data-v-bb58b91c=""><div class="companyListing__collectionTopHeading" data-v-bb58b91c="">
					Top Companies in
				</div> <div class="companyListing__collectionIndiaHeading" data-v-bb58b91c=""><img data-v-bb58b91c="" height="12" src="https://static.ambitionbox.com/static/loc/loc-star.png" width="12"/>
					INDIA
					<img data-v-bb58b91c="" height="12" src="https://static.ambitionbox.com/static/loc/loc-star.png" width="12"/></div></h1>

In [23]:
#To get the text of h1 tag
soup.find_all('h1')[0].text

'\n\t\t\t\t\tTop Companies in\n\t\t\t\t \n\t\t\t\t\tINDIA\n\t\t\t\t\t'

In [24]:
soup.find_all('h2')

[<h2 class="companyListing__title">
 								Companies in India
 							</h2>,
 <h2 class="companyCardWrapper__companyName" title="TCS">
 									TCS
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="Accenture">
 									Accenture
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="Wipro">
 									Wipro
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="Cognizant">
 									Cognizant
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="Capgemini">
 									Capgemini
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="HDFC Bank">
 									HDFC Bank
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="Infosys">
 									Infosys
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="HCLTech">
 									HCLTech
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="ICICI Bank">
 									ICICI Bank
 								</h2>,
 <h2 class="companyCardWrapper__companyName" ti

In [25]:
# soup.find_all('h2')[0].text.strip() ## -- Printing all company names using a for loop

for i in soup.find_all('h2'):
    print(i.text.strip()) ##Company names scraped from webpage-1

Companies in India
TCS
Accenture
Wipro
Cognizant
Capgemini
HDFC Bank
Infosys
HCLTech
ICICI Bank
Tech Mahindra
Genpact
TP
Axis Bank
Jio
Concentrix Corporation
Amazon
Reliance Retail
iEnergizer
LTM Limited
HDB Financial Services
Popular Collections by Industries
Popular Collections by Cities
Popular Collections by Roles


In [26]:
# Now along with company names, there are also some extra outputs like Popular Collections by Industries etc..
# use class attribute of soup.find_all

soup.find_all('h2', class_= 'companyCardWrapper__companyName') #see the class which the h2 tag belongs too

print(len(soup.find_all('h2', class_= 'companyCardWrapper__companyName'))) #20 companies in 1 webpage

for i in soup.find_all('h2', class_= 'companyCardWrapper__companyName'):
    print(i.text.strip())


20
TCS
Accenture
Wipro
Cognizant
Capgemini
HDFC Bank
Infosys
HCLTech
ICICI Bank
Tech Mahindra
Genpact
TP
Axis Bank
Jio
Concentrix Corporation
Amazon
Reliance Retail
iEnergizer
LTM Limited
HDB Financial Services


In [27]:
# Similarly we can extract ratings and reviews, but if we try to extract headquarter location and type of company,
# We can see on the html code of webpage1 that they have the same class!
# So, to fix this issue instead of going feature by feature through many loops, i will first extract the individual company cards
# and then will loop inside that company card

In [28]:
# We will first extract company cards and then we will loop inside it..
# The features which i need are present in the primary info div --

soup.find_all('div', class_= 'companyCardWrapper__primaryInformation')[0]

# This is the container for the first company, contains all the features i need for dataset

<div class="companyCardWrapper__primaryInformation"><div class="companyCardWrapper__companyLogo"><img alt="Tata Consultancy Services logo" height="50" loading="lazy" onerror="this.onerror=null;this.src='/static/icons/company-placeholder.svg';" src="https://static.ambitionbox.com/assets/v2/images/rs:fit:200:200:false:false/aHR0cHM6Ly9tZWRpYS5uYXVrcmkuY29tL21lZGlhL2FiY29tcGxvZ28vdGNzLW9yaWdpbmFsLmpwZw.webp" width="50"/></div> <div class="companyCardWrapper__metaInformation"><div class="companyCardWrapper__header"><div class="companyCardWrapper__companyPrimaryDetailsTopSection"><a class="companyCardWrapper__companyName"><h2 class="companyCardWrapper__companyName" title="TCS">
									TCS
								</h2></a> <!-- --></div> <button arialabel="Follow" class="companyCardWrapper__FollowCTA g-btn g-btn--text g-btn--md" data-testid="" title="Follow" type="button"><!-- --> <span class="g-btn__label g-btn__label--md g-btn__label--text g-btn__label--loader">Follow</span> <!-- --> <!-- --></button> <

In [29]:
##NOTE:
# Difference between soup.find and soup.find_all--

### -- soup.find is used when we only have one tag with same class..
### -- soup.find_all is used when we have multiple tags with same class.. so we get a list [look at the loop above]

In [30]:
names = []
ratings = []
num_reviews = []
high_rated = []
critical_rated = []
headquarter_loc = []
company_type = []

company = soup.find_all('div', class_= 'companyCardWrapper__primaryInformation')

for i in company:

    names.append(i.find('h2', class_= 'companyCardWrapper__companyName').text.strip())
    ratings.append(i.find('div', class_='rating_text').text.strip())
    num_reviews.append(i.find('span', class_='companyCardWrapper__companyRatingCount').text.strip())


#Code to extract highly rated for and critically rated for
    rating_labels = i.find_all('span', class_= 'companyCardWrapper__ratingValues')

    high = None
    critical = None

    for span in rating_labels:

        parent_text = span.parent.text.upper()

        if 'HIGHLY RATED' in parent_text:
            high = span.text.strip()
        elif 'CRITICALLY RATED' in parent_text:
            critical = span.text.strip()
    high_rated.append(high)
    critical_rated.append(critical)

#Code to extract company_type and headquarter_loc

    type_and_loc = i.find_all('span', class_= 'companyCardWrapper__interLinking')

    for span in type_and_loc:

        text_split = span.text.strip().split('|')

        type_comp = text_split[0].strip()
        loc = text_split[1].strip()

    company_type.append(type_comp)
    headquarter_loc.append(loc)
        


print(names)
print(ratings)
print(num_reviews)
print(high_rated)
print(critical_rated)
print(company_type)
print(headquarter_loc)

# All 7 features extracted from page1 of ambition box

['TCS', 'Accenture', 'Wipro', 'Cognizant', 'Capgemini', 'HDFC Bank', 'Infosys', 'HCLTech', 'ICICI Bank', 'Tech Mahindra', 'Genpact', 'TP', 'Axis Bank', 'Jio', 'Concentrix Corporation', 'Amazon', 'Reliance Retail', 'iEnergizer', 'LTM Limited', 'HDB Financial Services']
['3.3', '3.7', '3.6', '3.7', '3.6', '3.8', '3.5', '3.4', '3.9', '3.3', '3.6', '3.9', '3.6', '4.4', '3.5', '3.9', '3.9', '4.6', '3.6', '3.9']
['(1.2L)', '(75.3k)', '(66.4k)', '(62.7k)', '(54.8k)', '(53.9k)', '(49.9k)', '(47.1k)', '(46.8k)', '(44.3k)', '(43.5k)', '(39.7k)', '(34.1k)', '(34.1k)', '(32.9k)', '(32.4k)', '(27.8k)', '(27.5k)', '(27k)', '(26.3k)']
['Job Security', None, None, None, 'Work Life Balance, Job Security', 'Job Security, Skill Development', 'Job Security', None, 'Job Security, Promotions, Skill Development', None, 'Job Security', 'Work Life Balance, Job Security, Skill Development', None, 'Job Security, Skill Development, Company Culture', 'Job Security', 'Company Culture, Work Life Balance, Salary', 'J

In [31]:
names = []
ratings = []
num_reviews = []
high_rated = []
critical_rated = []
headquarter_loc = []
company_type = []

In [32]:
# Forming the final dataset

for i in range(1, 501):

    headers = headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36'}
    webpage = requests.get(f"https://www.ambitionbox.com/list-of-companies?page={i}", headers=headers).text

    soup = BeautifulSoup(webpage, "lxml")

    company = soup.find_all("div", class_="companyCardWrapper__primaryInformation")

    for j in company:

        names.append(j.find('h2', class_= 'companyCardWrapper__companyName').text.strip())

        # Handling logic for companies which dont have rating data
        ratings_extract = j.find('div', class_='rating_text')

        if(ratings_extract):
            ratings.append(ratings_extract.text.strip())
        else:
            ratings.append(None)

        # Handling logic for companies who dont have reviews
        reviews_extract = j.find('span', class_='companyCardWrapper__companyRatingCount')
        
        if(reviews_extract):
            num_reviews.append(reviews_extract.text.strip())
        else:
            num_reviews.append(None)

        #Code to extract highly rated for and critically rated for
        rating_labels = j.find_all('span', class_= 'companyCardWrapper__ratingValues')

        high = None
        critical = None

        for span in rating_labels:

            parent_text = span.parent.text.upper()

            if 'HIGHLY RATED' in parent_text:
                high = span.text.strip()
            elif 'CRITICALLY RATED' in parent_text:
                critical = span.text.strip()
        high_rated.append(high)
        critical_rated.append(critical)

        #Code to extract company_type and headquarter_loc

        type_and_loc = j.find_all('span', class_= 'companyCardWrapper__interLinking')

        for span in type_and_loc:

            text_split = span.text.strip().split('|')

            # type_comp = text_split[0].strip()
            # loc = text_split[1].strip() //Here if some company doesnt have loc or type, then index error might come
            
            type_comp = None
            loc = None

            if(len(text_split) > 0):
                type_comp = text_split[0].strip()
            
            if(len(text_split) > 1):
                loc = text_split[1].strip()

        company_type.append(type_comp)
        headquarter_loc.append(loc)
    

In [33]:
names = {"Name": names, "Rating": ratings, "Type": company_type, "Num_of_Reviews": num_reviews, "Highly Rated for": high_rated, "Critically Rated for": critical_rated, "Location": headquarter_loc}

df = pd.DataFrame(names)

df

,Name,Rating,Type,Num_of_Reviews,Highly Rated for,Critically Rated for,Location
0,TCS,3.3,IT Services & Consulting,(1.2L),Job Security,"Promotions, Salary, Work Satisfaction",Bengaluru +448 other locations
1,Accenture,3.7,IT Services & Consulting,(75.3k),NaN,"Promotions, Salary, Work Satisfaction",Bengaluru +261 other locations
2,Wipro,3.6,IT Services & Consulting,(66.4k),NaN,"Promotions, Salary, Work Satisfaction",Hyderabad +375 other locations
3,Cognizant,3.7,IT Services & Consulting,(62.7k),NaN,"Promotions, Salary, Work Satisfaction",Hyderabad +233 other locations
4,Capgemini,3.6,IT Services & Consulting,(54.8k),"Work Life Balance, Job Security","Promotions, Salary, Work Satisfaction",Bengaluru +188 other locations
...,...,...,...,...,...,...,...
9995,Poonam IT Consulting Services,4.1,Recruitment,(113),"Company Culture, Skill Development, Work Life ...",NaN,Bengaluru +10 other locations
9996,The World Bank,3.8,Banking,(113),"Work Life Balance, Salary","Promotions, Job Security",Chennai +12 other locations
9997,Allegion,4.2,Law Enforcement & Security,(113),"Work Life Balance, Company Culture, Job Security",NaN,Bengaluru +2 other locations
9998,Shigan Quantum Technologies,3.9,Auto Components,(113),Work Life Balance,"Promotions, Salary",Gurugram +8 other locations


In [34]:
df.shape

(10000, 7)

In [35]:
df.to_csv("../Fetching data about companies using Web-Scraping/company_data.csv", index=False)